# SECURE TABULAR DATA GENERATION PLATFORM
### BẢNG ĐIỀU KHIỂN & HỆ THỐNG THỬ NGHIỆM SẢN PHẨM HOÀN CHỈNH (END-TO-END DEMO)

Notebook này đóng vai trò như một **giao diện sản phẩm hoàn chỉnh** đang trong quá trình thử nghiệm nghiệm thu. 
Bạn có thể thay đổi cấu hình dữ liệu, mô hình sinh, và thiết lập an toàn Differential Privacy trực tiếp tại bảng điều khiển cấu hình bên dưới và chạy toàn bộ hệ thống từ đầu đến cuối.

## BƯỚC 1: BẢNG CẤU HÌNH HỆ THỐNG (CONTROL PANEL)
Hãy thay đổi các giá trị biến dưới đây tùy theo kịch bản thuyết trình của bạn, sau đó chạy ô này.

In [ ]:
# ==========================================================================
#                  BẢNG CẤU HÌNH HỆ THỐNG THỬ NGHIỆM
# ==========================================================================

# 1. Lựa chọn tập dữ liệu nguồn (Source Dataset)
# Cấu hình: "telco_customer_churn" (Dữ liệu khách hàng) hoặc "adult_income" (Dữ liệu thu nhập)
dataset_name = "telco_customer_churn"

# 2. Lựa chọn kiến trúc mô hình học sâu sinh dữ liệu (Generative Model)
# Cấu hình: "diffusion" (Tabular Diffusion), "ctvae" (Conditional VAE), hoặc "ctgan" (Conditional GAN)
model_type = "diffusion"

# 3. Thiết lập cơ chế Bảo mật vi sai (Differential Privacy)
enable_dp = True

# 4. Ngân sách bảo mật Epsilon (Privacy Budget)
# Cấu hình: 10.0 (Hữu dụng cao), 3.0 (Cân bằng), hoặc 1.5 (Bảo mật tối đa)
epsilon = 3.0

# 5. Số lượng dòng dữ liệu tổng hợp cần sinh ra (Synthetic Rows)
n_rows = 1000

# 6. Chế độ chạy mô phỏng nghiệm thu nhanh hay huấn luyện thật
# Cấu hình: "simulation" (Nạp checkpoint chạy mất 3 giây), "training" (Huấn luyện lại từ đầu mất 5-15 phút)
execution_mode = "simulation"

# ==========================================================================

import os
import sys
project_root = os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

print("=== CẤU HÌNH ĐÃ GHI NHẬN ===")
print(f"  * Dataset    : {dataset_name.upper()}")
print(f"  * Model      : {model_type.upper()}")
print(f"  * DP Active  : {enable_dp} (Epsilon = {epsilon if enable_dp else 'inf'})")
print(f"  * Mode       : {execution_mode.upper()}")
print(f"  * Rows       : {n_rows}")

## BƯỚC 2: KHỞI CHẠY PIPELINE & SINH DỮ LIỆU TỔNG HỢP
Chạy ô này để thực hiện thu thập, làm sạch dữ liệu, mã hóa, chạy mô hình sinh và áp dụng bộ kiểm soát ràng buộc nghiệp vụ cứng (`ConstraintsEngine`).

In [ ]:
from src.config.config_loader import ConfigLoader
from src.preprocessing.pipeline import PreprocessingPipeline
from src.inference.sampler import SyntheticSampler
from src.training.trainer import ModelTrainer, set_global_seed

set_global_seed(42)

# 1. Giải quyết đường dẫn dữ liệu thô
if dataset_name == "telco_customer_churn":
    data_path = os.path.join("data", "Telco-Customer-Churn.csv")
else:
    data_path = os.path.join("data", "adult", "adult.data")

print("[1] Đang nạp cấu hình và tiền xử lý dữ liệu (Stage 1 & 2)...")
config = ConfigLoader.load_config(dataset_name)
pipeline = PreprocessingPipeline(dataset_name)
df_raw = pipeline.load_data(data_path)
df_preprocessed = pipeline.fit_transform(df_raw)
pipeline.save_artifacts()
print(f"    Dữ liệu thô: {df_raw.shape} | Dữ liệu sau tiền xử lý: {df_preprocessed.shape}")

# 2. Sinh dữ liệu tổng hợp (Stage 3 & 4)
if execution_mode == "simulation":
    print("\n[2] Chạy chế độ mô phỏng: Nạp checkpoint đã huấn luyện từ trước...")
    sampler = SyntheticSampler(model_type=model_type, dataset_name=dataset_name)
    sampler.load()
    df_synthetic = sampler.generate(n_rows=n_rows)
else:
    print("\n[2] Huấn luyện mô hình thực tế trong 100 epochs (vui lòng đợi GPU/CPU)...")
    trainer = ModelTrainer(model_type=model_type, dataset_name=dataset_name)
    trainer.train(
        preprocessed_df=df_preprocessed,
        continuous_cols=pipeline.continuous_cols,
        categorical_cols=pipeline.categorical_cols,
        epochs=100,
        batch_size=256,
        lr=1e-3,
        seed=42,
        enable_dp=enable_dp,
        target_epsilon=epsilon,
        target_delta=1e-5
    )
    sampler = SyntheticSampler(model_type=model_type, dataset_name=dataset_name)
    sampler.load()
    df_synthetic = sampler.generate(n_rows=n_rows)

print(f"\nSinh dữ liệu thành công! Kích thước dữ liệu tổng hợp: {df_synthetic.shape}")
display(df_synthetic.head())

## BƯỚC 3: CHẠY BỘ KIỂM TOÁN CHẤT LƯỢNG ĐA CHIỀU (AUDIT ENGINE)
Chạy ô này để thực hiện đánh giá Độ trung thực (Fidelity), Bảo mật (Privacy - MIA, DCR) và Độ hữu dụng học máy (Utility - TSTR vs TRTR) của dữ liệu tổng hợp sinh ra.

In [ ]:
from src.evaluation.orchestrator import EvaluationSuite

print("[3] Khởi chạy kiểm toán chất lượng đa chiều...")
suite = EvaluationSuite(dataset_name=dataset_name)

target_col = config.ingestion.target_column
sensitive_col = config.ingestion.quasi_identifiers[0] if config.ingestion.quasi_identifiers else ""

# Chạy kiểm định
results = suite.run_evaluation(
    real_df=df_raw,
    synth_df=df_synthetic,
    target_col=target_col,
    sensitive_col=sensitive_col
)

print("\n" + "="*70)
print(f"        KẾT QUẢ KIỂM TOÁN CHẤT LƯỢNG MÔ HÌNH {model_type.upper()}")
print("="*70)
print(f"1. FIDELITY (Độ trung thực thống kê):")
print(f"   - JSD biên trung bình (Avg JSD)      : {results['fidelity']['avg_js']:.4f} (càng gần 0 càng tốt)")
print(f"   - Lệch tương quan chéo (Corr Diff)    : {results['fidelity']['corr_diff']:.4f} (càng gần 0 càng tốt)")
print(f"\n2. PRIVACY (An toàn bảo mật):")
print(f"   - Tấn công thành viên (MIA AUC-ROC) : {results['privacy']['mia_auc']:.4f} (gần 0.50 là bảo mật tốt nhất)")
print(f"   - Tỷ lệ rò rỉ khoảng cách gần (DCR)   : {results['privacy']['dcr_leakage_pct']:.2f}%")
print(f"   - Khoảng cách Euclid trung bình (Mean): {results['privacy']['dcr_mean']:.4f}")
print(f"\n3. UTILITY (Độ hữu dụng cho Machine Learning - TSTR vs TRTR):")
for clf, metrics in results['utility']['metrics'].items():
    print(f"   - {clf:<20} : TSTR F1 = {metrics['tstr_f1']:.4f} (Thật TRTR = {metrics['trtr_f1']:.4f})")
print("="*70)

## BƯỚC 4: TỰ ĐỘNG KẾT XUẤT VÀ HIỂN THỊ BÁO CÁO HTML
Chạy ô này để hệ thống **tự động mở Báo cáo Kiểm toán Tuân thủ (Compliance Report) dạng HTML** trên trình duyệt web của bạn, đồng thời vẽ các biểu đồ phân phối đè phủ bên dưới.

In [ ]:
import webbrowser

html_report_path = results['report_paths']['html']
print(f"Đường dẫn báo cáo HTML: {html_report_path}")

# Tự động mở trình duyệt web
try:
    webbrowser.open(f"file:///{html_report_path}")
    print("\n[✓] Hệ thống đã tự động mở báo cáo HTML thành công trên trình duyệt của bạn!")
except Exception as e:
    print("\n[!] Không thể tự động mở. Bạn hãy mở thủ công đường dẫn trên.")

# Vẽ biểu đồ trực tiếp trong Notebook
plots_dir = suite.plots_dir
corr_plot = os.path.join(plots_dir, "correlation_comparison.png")
if os.path.exists(corr_plot):
    plt.figure(figsize=(15, 8))
    img = plt.imread(corr_plot)
    plt.imshow(img)
    plt.axis('off')
    plt.title("So sánh Tương Quan Chéo (Real vs Synthetic)", fontsize=14, fontweight='bold')
    plt.show()

dist_plot = os.path.join(plots_dir, "distributions_grid.png")
if os.path.exists(dist_plot):
    plt.figure(figsize=(15, 12))
    img = plt.imread(dist_plot)
    plt.imshow(img)
    plt.axis('off')
    plt.title("Đè phủ Phân Phối Biên (Real vs Synthetic)", fontsize=14, fontweight='bold')
    plt.show()